データ読み込み

In [1]:
import pandas as pd
import re
import ifcopenshell

# CSV ファイルのパス
csv_path = r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\resources\BDNS_Abbreviations_Register_v2.0.0.csv"

# CSV の読み込み
df = pd.read_csv(csv_path)

# データ構造の確認
df.head()

,asset_description,asset_abbreviation,can_be_connected,dbo_entity_type,ifc4_3,ifc2x3
0,access control - RFID controller,RFIDC,1,NaN,IfcController,IfcController
1,access control - RFID reader,RFIDR,1,NaN,IfcCommunicationsApplianceSCANNER,IfcElectricApplianceSCANNER
2,access control - access control system,ACS,1,NaN,IfcDistributionSystemSECURITY,NaN
3,access control - audio intercom,AIC,1,NaN,IfcSwitchingDevice,IfcSwitchingDevice
4,access control - biometric reader,BIOR,1,NaN,IfcCommunicationsApplianceSCANNER,IfcElectricApplianceSCANNER


ifcのカバー範囲抽出

In [2]:
ifc4_3_unique = (
    df["ifc4_3"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

ifc4_3_unique_sorted = sorted(ifc4_3_unique)

ifc4_3_unique_sorted

['IfcActuator',
 'IfcActuatorELECTRICACTUATOR',
 'IfcAirTerminal',
 'IfcAirTerminalBox',
 'IfcAirTerminalBoxCONSTANTFLOW',
 'IfcAirToAirHeatRecovery',
 'IfcAirToAirHeatRecoveryROTARYWHEEL',
 'IfcAirToAirHeatRecoveryRUNAROUNDCOIL',
 'IfcAlarm',
 'IfcAlarmBELL',
 'IfcAlarmBREAKGLASSBUTTON',
 'IfcAlarmLIGHT',
 'IfcAlarmMANUALPULLBOX',
 'IfcAlarmSIREN',
 'IfcAudioVisualAppliance',
 'IfcAudioVisualApplianceAMPLIFIER',
 'IfcAudioVisualApplianceCAMERA',
 'IfcAudioVisualApplianceCOMMUNICATIONTERMINAL',
 'IfcAudioVisualApplianceDISPLAY',
 'IfcAudioVisualApplianceMICROPHONE',
 'IfcAudioVisualAppliancePLAYER',
 'IfcAudioVisualApplianceRECEIVER',
 'IfcAudioVisualApplianceSPEAKER',
 'IfcAudioVisualApplianceSWITCHER',
 'IfcAudioVisualApplianceTUNER',
 'IfcBoiler',
 'IfcBoilerSTEAM',
 'IfcBoilerWATER',
 'IfcBurner',
 'IfcCableCarrierSegment',
 'IfcCableCarrierSegmentCABLELADDERSEGMENT',
 'IfcCableCarrierSegmentCABLETRAYSEGMENT',
 'IfcCableCarrierSegmentCABLETRUNKINGSEGMENT',
 'IfcCableSegment',
 'Ifc

In [3]:
ifc2x3_unique = (
    df["ifc2x3"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

ifc2x3_unique_sorted = sorted(ifc2x3_unique)

ifc2x3_unique_sorted

['IfcActuator',
 'IfcActuatorELECTRICACTUATOR',
 'IfcAirTerminal',
 'IfcAirTerminalBox',
 'IfcAirTerminalBoxCONSTANTFLOW',
 'IfcAirToAirHeatRecovery',
 'IfcAirToAirHeatRecoveryROTARYWHEEL',
 'IfcAirToAirHeatRecoveryRUNAROUNDCOIL',
 'IfcAlarm',
 'IfcAlarmBELL',
 'IfcAlarmBREAKGLASSBUTTON',
 'IfcAlarmLIGHT',
 'IfcAlarmMANUALPULLBOX',
 'IfcAlarmSIREN',
 'IfcBoiler',
 'IfcBoilerSTEAM',
 'IfcBoilerWATER',
 'IfcBuildingElementProxy',
 'IfcCableCarrierSegment',
 'IfcCableCarrierSegmentCABLELADDERSEGMENT',
 'IfcCableCarrierSegmentCABLETRAYSEGMENT',
 'IfcCableCarrierSegmentCABLETRUNKINGSEGMENT',
 'IfcCableSegment',
 'IfcCableSegmentFIBERTUBE',
 'IfcChiller',
 'IfcChillerAIRCOOLED',
 'IfcChillerWATERCOOLED',
 'IfcCoil',
 'IfcCoilDXCOOLINGCOIL',
 'IfcCoilHYDRONICCOIL',
 'IfcCoilWATERCOOLINGCOIL',
 'IfcCoilWATERHEATINGCOIL',
 'IfcCompressor',
 'IfcCondenser',
 'IfcCondenserAIRCOOLED',
 'IfcController',
 'IfcControllerTWOPOSITION',
 'IfcCooledBeam',
 'IfcCoolingTower',
 'IfcDamper',
 'IfcDamperBALA

Entity と PredefinedType を分解する関数定義

In [4]:
def split_ifc_entity_and_type(value):
    if pd.isna(value):
        return pd.Series([None, None])

    value = str(value)

    match = re.match(r"^(Ifc[A-Za-z0-9]+?)([A-Z]+)$", value)

    if match:
        entity = match.group(1)
        predefined_type = match.group(2)
        return pd.Series([entity, predefined_type])
    else:
        return pd.Series([value, None])

In [5]:
df[["ifc2x3_entity", "ifc2x3_predefined_type"]] = (
    df["ifc2x3"]
    .apply(split_ifc_entity_and_type)
)

df[["ifc2x3", "ifc2x3_entity", "ifc2x3_predefined_type"]].head()

,ifc2x3,ifc2x3_entity,ifc2x3_predefined_type
0,IfcController,IfcController,None
1,IfcElectricApplianceSCANNER,IfcElectricAppliance,SCANNER
2,NaN,None,None
3,IfcSwitchingDevice,IfcSwitchingDevice,None
4,IfcElectricApplianceSCANNER,IfcElectricAppliance,SCANNER


In [6]:
df[["ifc4_3_entity", "ifc4_3_predefined_type"]] = (
    df["ifc4_3"]
    .apply(split_ifc_entity_and_type)
)

df[["ifc4_3", "ifc4_3_entity", "ifc4_3_predefined_type"]].head()

,ifc4_3,ifc4_3_entity,ifc4_3_predefined_type
0,IfcController,IfcController,None
1,IfcCommunicationsApplianceSCANNER,IfcCommunicationsAppliance,SCANNER
2,IfcDistributionSystemSECURITY,IfcDistributionSystem,SECURITY
3,IfcSwitchingDevice,IfcSwitchingDevice,None
4,IfcCommunicationsApplianceSCANNER,IfcCommunicationsAppliance,SCANNER


IFC要素抽出プログラム

In [7]:
import ifcopenshell

def extract_ifc_elements(ifc_path):
    model = ifcopenshell.open(ifc_path)

    elements = []

    for obj in model.by_type("IfcObject"):
        entity_name = obj.is_a()

        predefined_type = None
        if hasattr(obj, "PredefinedType"):
            predefined_type = obj.PredefinedType

        elements.append({
            "global_id": obj.GlobalId,
            "ifc_entity": entity_name,
            "predefined_type": None if predefined_type in [None, ""] else str(predefined_type)
        })

    return elements

In [8]:
ifc_elements = extract_ifc_elements(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\raw\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc")

判定関数

In [13]:
bdns_sub_df = df[["ifc2x3_entity", "ifc2x3_predefined_type"]]

In [16]:
bdns_exact_set = set(
    (
        row["ifc2x3_entity"],
        str(row["ifc2x3_predefined_type"]).upper()
    )
    for _, row in bdns_sub_df.iterrows()
    if not pd.isna(row["ifc2x3_predefined_type"])
)

In [17]:
bdns_exact_set

{('IfcActuator', 'ELECTRICACTUATOR'),
 ('IfcAirTerminalBox', 'CONSTANTFLOW'),
 ('IfcAirToAirHeatRecovery', 'ROTARYWHEEL'),
 ('IfcAirToAirHeatRecovery', 'RUNAROUNDCOIL'),
 ('IfcAlarm', 'BELL'),
 ('IfcAlarm', 'BREAKGLASSBUTTON'),
 ('IfcAlarm', 'LIGHT'),
 ('IfcAlarm', 'MANUALPULLBOX'),
 ('IfcAlarm', 'SIREN'),
 ('IfcBoiler', 'STEAM'),
 ('IfcBoiler', 'WATER'),
 ('IfcCableCarrierSegment', 'CABLELADDERSEGMENT'),
 ('IfcCableCarrierSegment', 'CABLETRAYSEGMENT'),
 ('IfcCableCarrierSegment', 'CABLETRUNKINGSEGMENT'),
 ('IfcCableSegment', 'FIBERTUBE'),
 ('IfcChiller', 'AIRCOOLED'),
 ('IfcChiller', 'WATERCOOLED'),
 ('IfcCoil', 'DXCOOLINGCOIL'),
 ('IfcCoil', 'HYDRONICCOIL'),
 ('IfcCoil', 'WATERCOOLINGCOIL'),
 ('IfcCoil', 'WATERHEATINGCOIL'),
 ('IfcCondenser', 'AIRCOOLED'),
 ('IfcController', 'TWOPOSITION'),
 ('IfcDamper', 'BALANCINGDAMPER'),
 ('IfcDamper', 'CONTROLDAMPER'),
 ('IfcDamper', 'FIREDAMPER'),
 ('IfcDamper', 'FIRESMOKEDAMPER'),
 ('IfcDamper', 'FUMEHOODEXHAUST'),
 ('IfcDamper', 'RELIEFDAMPER

In [18]:
bdns_entity_only_set = set(
    row["ifc2x3_entity"]
    for _, row in bdns_sub_df.iterrows()
    if pd.isna(row["ifc2x3_predefined_type"])
)

In [20]:
def is_defined_in_bdns(ifc_element,
                       bdns_exact_set,
                       bdns_entity_only_set):

    entity = ifc_element["ifc_entity"]
    ptype = ifc_element["predefined_type"]

    if entity in bdns_entity_only_set:
        return True

    if ptype is None:
        return False

    return (entity, ptype.upper()) in bdns_exact_set

In [21]:
for elem in ifc_elements:
    elem["allow_bdns_tag"] = is_defined_in_bdns(
        elem,
        bdns_exact_set,
        bdns_entity_only_set
    )

In [22]:
ifc_elements

[{'global_id': '3T5NVXezz9DRBS$A4Fkvli',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvkk',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvku',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvkf',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvks',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvkr',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '0jKcM_OSPB4BmhGqdswtba',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},
 {'global_id': '3T5NVXezz9DRBS$A4Fkvkh',
  'ifc_entity': 'IfcSystem',
  'predefined_type': None,
  'allow_bdns_tag': False},


In [26]:
bdns_true_elements = [
    elem for elem in ifc_elements
    if elem.get("allow_bdns_tag", False)
]

In [27]:
bdns_true_elements

[{'global_id': '3edPp6VUPFUPmy7DmdPW$P',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3edPp6VUPFUPmy7DmdP$bi',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3edPp6VUPFUPmy7DmdPzeS',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3edPp6VUPFUPmy7DmdPztO',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '1zT1vkSdP5geic0HutmWkl',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3ThYxDUCD5tewfq2Thu9$0',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3ThYxDUCD5tewfq2Thu9Xh',
  'ifc_entity': 'IfcBuildingElementProxy',
  'predefined_type': None,
  'allow_bdns_tag': True},
 {'global_id': '3ThYxDUCD5tewfq2Th

In [28]:
bdns_true_count = len(bdns_true_elements)

In [29]:
bdns_true_count

223